# Independent CPU readback: local shards0031 and0032
69 unchanged historical Main anchors and40 new local jobs; count the anchors once across capsules. No training or Final labels.

In [ ]:
import hashlib
import importlib.util
import json
import platform
import time
from pathlib import Path

import torch

CAPSULE_SHA = "78449c917ebea4d719042c5849b095bf73f44a1f519cce1bbe86e0397df69675"
HELPER_SHA = "78c4bd421ec0971023dca141ed8e81b22dbc97fe5c5855cd2511bfb30d1761be"
assert not torch.cuda.is_available(), "This independent readback must use CPU"
archives = list(Path("/kaggle/input").rglob("review-capsule.zip.bin"))
assert len(archives) == 1, "Attach the exact private capsule Dataset version1"
archive = archives[0]
helper = archive.parent / "hung_review_capsule.py"
assert hashlib.sha256(helper.read_bytes()).hexdigest() == HELPER_SHA
spec = importlib.util.spec_from_file_location("hung_review_capsule", helper)
capsule = importlib.util.module_from_spec(spec)
spec.loader.exec_module(capsule)
started = time.perf_counter()
result = capsule.verify(archive, CAPSULE_SHA, score=True)
assert result["passed"] and result["main_runs"] == 69 and result["new_local_jobs"] == 40
assert len(result["computed_runs"]) == 109 and result["scored_runs"] == 59
result.update(
    {
        "elapsed_seconds": time.perf_counter() - started,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": "cpu",
        "helper_sha256": HELPER_SHA,
        "purpose": "Independent private Dataset readback and saved Development forecast scoring; no training or Final",
    }
)
with Path("/kaggle/working/readback.json").open("x") as stream:
    json.dump(result, stream, indent=2, sort_keys=True, allow_nan=False)
summary = {k: v for k, v in result.items() if k != "computed_runs"}
with Path("/kaggle/working/readback-summary.json").open("x") as stream:
    json.dump(summary, stream, indent=2, sort_keys=True, allow_nan=False)
print(json.dumps(summary), flush=True)